In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [ ]:
#hide
from fastbook import *

# Un modello linguistico da zero

Ora siamo pronti per andare in profondità... in profondità nel deep learning! Hai già imparato come addestrare una rete neurale di base, ma come passare da lì alla creazione di modelli all'avanguardia? In questa parte del libro sveleremo tutti i misteri, a cominciare dai modelli linguistici.

Hai visto in <<chapter_nlp>> come mettere a punto un modello linguistico preaddestrato per creare un classificatore di testo. In questo capitolo ti spiegheremo cosa c'è esattamente all'interno di quel modello e cos'è una RNN. Innanzitutto, raccogliamo alcuni dati che ci consentiranno di prototipare rapidamente i nostri vari modelli.

## I dati

Ogni volta che iniziamo a lavorare su un nuovo problema, proviamo sempre prima a pensare al set di dati più semplice possibile che ci consentirà di provare i metodi in modo rapido e semplice e di interpretare i risultati. Quando abbiamo iniziato a lavorare sulla modellazione del linguaggio qualche anno fa, non abbiamo trovato alcun set di dati che consentisse una prototipazione rapida, quindi ne abbiamo creato uno. Lo chiamiamo *Numeri Umani* e contiene semplicemente i primi 10.000 numeri scritti in inglese.

> j: Uno degli errori pratici più comuni che vedo anche tra professionisti di grande esperienza è non utilizzare set di dati appropriati nei momenti appropriati durante il processo di analisi. In particolare, la maggior parte delle persone tende a iniziare con set di dati troppo grandi e troppo complicati.

Possiamo scaricare, estrarre e dare un'occhiata al nostro set di dati nel solito modo:

In [ ]:
from fastai.text.all import *
path = untar_data(URLs.HUMAN_NUMBERS)

In [ ]:
#hide
Path.BASE_PATH = path

In [ ]:
path.ls()

(#2) [Path('train.txt'),Path('valid.txt')]

Apriamo questi due file e vediamo cosa c'è dentro. Inizialmente uniremo tutti i testi insieme e ignoreremo la suddivisione treno/valido fornita dal set di dati (torneremo su questo più tardi):

In [ ]:
lines = L()
with open(path/'train.txt') as f: lines += L(*f.readlines())
with open(path/'valid.txt') as f: lines += L(*f.readlines())
lines

(#9998) ['one \n','two \n','three \n','four \n','five \n','six \n','seven \n','eight \n','nine \n','ten \n'...]

Prendiamo tutte quelle righe e le concateniamo in un unico grande flusso. Per contrassegnare quando si passa da un numero al successivo, utilizziamo un `.` come separatore:

In [ ]:
text = ' . '.join([l.strip() for l in lines])
text[:100]

'one . two . three . four . five . six . seven . eight . nine . ten . eleven . twelve . thirteen . fo'

Possiamo tokenizzare questo set di dati suddividendolo negli spazi:

In [ ]:
tokens = text.split(' ')
tokens[:10]

['one', '.', 'two', '.', 'three', '.', 'four', '.', 'five', '.']

Per numerizzare, dobbiamo creare un elenco di tutti i token univoci (il nostro *vocabolario*):

In [ ]:
vocab = L(*tokens).unique()
vocab

(#30) ['one','.','two','three','four','five','six','seven','eight','nine'...]

Quindi possiamo convertire i nostri token in numeri cercando l'indice di ciascuno nel vocabolario:

In [ ]:
word2idx = {w:i for i,w in enumerate(vocab)}
nums = L(word2idx[i] for i in tokens)
nums

(#63095) [0,1,2,1,3,1,4,1,5,1...]

Ora che disponiamo di un piccolo set di dati su cui la modellazione del linguaggio dovrebbe essere un compito facile, possiamo costruire il nostro primo modello.

## Il nostro modello di prima lingua da zero

Un modo semplice per trasformarlo in una rete neurale sarebbe specificare che prediremo ogni parola in base alle tre parole precedenti. Potremmo creare un elenco di ogni sequenza di tre parole come variabili indipendenti e la parola successiva dopo ciascuna sequenza come variabile dipendente. 

Possiamo farlo con il semplice Python. Facciamolo prima con i token solo per confermare come appare:

In [ ]:
L((tokens[i:i+3], tokens[i+3]) for i in range(0,len(tokens)-4,3))

(#21031) [(['one', '.', 'two'], '.'),(['.', 'three', '.'], 'four'),(['four', '.', 'five'], '.'),(['.', 'six', '.'], 'seven'),(['seven', '.', 'eight'], '.'),(['.', 'nine', '.'], 'ten'),(['ten', '.', 'eleven'], '.'),(['.', 'twelve', '.'], 'thirteen'),(['thirteen', '.', 'fourteen'], '.'),(['.', 'fifteen', '.'], 'sixteen')...]

Ora lo faremo con i tensori dei valori numerici, che è ciò che effettivamente utilizzerà il modello:

In [ ]:
seqs = L((tensor(nums[i:i+3]), nums[i+3]) for i in range(0,len(nums)-4,3))
seqs

(#21031) [(tensor([0, 1, 2]), 1),(tensor([1, 3, 1]), 4),(tensor([4, 1, 5]), 1),(tensor([1, 6, 1]), 7),(tensor([7, 1, 8]), 1),(tensor([1, 9, 1]), 10),(tensor([10,  1, 11]), 1),(tensor([ 1, 12,  1]), 13),(tensor([13,  1, 14]), 1),(tensor([ 1, 15,  1]), 16)...]

Possiamo raggrupparli facilmente utilizzando la classe "DataLoader". Per ora divideremo le sequenze in modo casuale:

In [ ]:
bs = 64
cut = int(len(seqs) * 0.8)
dls = DataLoaders.from_dsets(seqs[:cut], seqs[cut:], bs=64, shuffle=False)

Ora possiamo creare un'architettura di rete neurale che accetta tre parole come input e restituisce una previsione della probabilità di ogni possibile parola successiva nel vocabolario. Utilizzeremo tre livelli lineari standard, ma con due modifiche.

La prima modifica è che il primo strato lineare utilizzerà solo l'incorporamento della prima parola come attivazioni, il secondo strato utilizzerà l'incorporamento della seconda parola più le attivazioni di output del primo strato e il terzo strato utilizzerà l'incorporamento della terza parola più le attivazioni di output del secondo strato. L'effetto chiave di ciò è che ogni parola viene interpretata nel contesto informativo di qualsiasi parola che la precede. 

La seconda modifica è che ciascuno di questi tre strati utilizzerà la stessa matrice di peso. Il modo in cui una parola influisce sulle attivazioni delle parole precedenti non dovrebbe cambiare a seconda della posizione di una parola. In altre parole, i valori di attivazione cambieranno man mano che i dati si spostano attraverso i livelli, ma i pesi dei livelli stessi non cambieranno da livello a livello. Pertanto, uno strato non apprende una posizione nella sequenza; deve imparare a gestire tutte le posizioni.

Poiché il peso dei livelli non cambia, potresti pensare ai livelli sequenziali come allo "stesso livello" ripetuto. In effetti, PyTorch lo rende concreto; possiamo semplicemente creare un livello e utilizzarlo più volte.

### Il nostro modello linguistico in PyTorch

Ora possiamo creare il modulo del modello linguistico che abbiamo descritto in precedenza:

In [ ]:
class LMModel1(Module):
    def __init__(self, vocab_sz, n_hidden):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)  
        self.h_h = nn.Linear(n_hidden, n_hidden)     
        self.h_o = nn.Linear(n_hidden,vocab_sz)
        
    def forward(self, x):
        h = F.relu(self.h_h(self.i_h(x[:,0])))
        h = h + self.i_h(x[:,1])
        h = F.relu(self.h_h(h))
        h = h + self.i_h(x[:,2])
        h = F.relu(self.h_h(h))
        return self.h_o(h)

Come vedi, abbiamo creato tre livelli:

- Il livello di incorporamento (`i_h`, da *input* a *hidden*)
- Il livello lineare per creare le attivazioni per la parola successiva (`h_h`, da *hidden* a *hidden*)
- Un livello lineare finale per prevedere la quarta parola (`h_o`, da *hidden* a *output*)

Potrebbe essere più semplice rappresentarlo in forma pittorica, quindi definiamo una semplice rappresentazione pittorica delle reti neurali di base. <<img_simple_nn>> mostra come rappresenteremo una rete neurale con uno strato nascosto.

<img alt="Rappresentazione pittorica di una rete neurale semplice" width="400" src="images/att_00020.png" caption="Rappresentazione pittorica di una rete neurale semplice" id="img_simple_nn">

Ogni forma rappresenta le attivazioni: rettangolo per l'input, cerchio per le attivazioni del livello nascosto (interno) e triangolo per le attivazioni dell'output. Utilizzeremo queste forme (riassunte in <<img_shapes>>) in tutti i diagrammi di questo capitolo.

<img alt="Forme utilizzate nelle nostre rappresentazioni pittoriche" width="200" src="images/att_00021.png" id="img_shapes" caption="Forme utilizzate nelle nostre rappresentazioni pittoriche">

Una freccia rappresenta il calcolo effettivo dello strato, ovvero lo strato lineare seguito dalla funzione di attivazione. Usando questa notazione, <<lm_rep>> mostra come appare il nostro semplice modello linguistico.

<img alt="Rappresentazione del nostro modello linguistico di base" width="500" caption="Rappresentazione del nostro modello linguistico di base" id="lm_rep" src="images/att_00022.png">

Per semplificare le cose, abbiamo rimosso i dettagli del calcolo dei livelli da ciascuna freccia. Abbiamo anche codificato i colori delle frecce, in modo tale che tutte le frecce con lo stesso colore abbiano la stessa matrice di peso. Ad esempio, tutti i livelli di input utilizzano la stessa matrice di incorporamento, quindi hanno tutti lo stesso colore (verde).

Proviamo ad addestrare questo modello e vediamo come va:

In [ ]:
learn = Learner(dls, LMModel1(len(vocab), 64), loss_func=F.cross_entropy, 
                metrics=accuracy)
learn.fit_one_cycle(4, 1e-3)

epoch,train_loss,valid_loss,accuracy,time
0,1.824297,1.970941,0.467554,00:02
1,1.386973,1.823242,0.467554,00:02
2,1.417556,1.654497,0.494414,00:02
3,1.376440,1.650849,0.494414,00:02


Per vedere se questo va bene, vediamo cosa ci darebbe un modello molto semplice. In questo caso potremmo sempre prevedere il token più comune, quindi scopriamo quale token è più spesso l'obiettivo nel nostro set di convalida:

In [ ]:
n,counts = 0,torch.zeros(len(vocab))
for x,y in dls.valid:
    n += y.shape[0]
    for i in range_of(vocab): counts[i] += (y==i).long().sum()
idx = torch.argmax(counts)
idx, vocab[idx.item()], counts[idx].item()/n

(tensor(29), 'thousand', 0.15165200855716662)

Il token più comune ha l'indice 29, che corrisponde al token "mille". Prevedere sempre questo token ci darebbe una precisione di circa il 15\%, quindi stiamo andando molto meglio!

> R: La mia prima ipotesi era che il separatore sarebbe stato il token più comune, poiché ce n'è uno per ogni numero. Ma guardando i "gettoni" mi sono ricordato che i grandi numeri si scrivono con molte parole, quindi sulla strada per 10.000 scrivi "mille" molto: cinquemila, cinquemilauno, cinquemiladue, ecc. Ops! Osservare i tuoi dati è ottimo per notare caratteristiche sottili e anche imbarazzanti.

Questa è una bella prima linea di base. Vediamo come possiamo rifattorizzarlo con un loop.

### La nostra prima rete neurale ricorrente

Osservando il codice del nostro modulo, potremmo semplificarlo sostituendo il codice duplicato che chiama i livelli con un ciclo "for". Oltre a rendere il nostro codice più semplice, questo avrà anche il vantaggio di poter applicare il nostro modulo altrettanto bene a sequenze di token di diversa lunghezza: non saremo limitati a elenchi di token di lunghezza tre:

In [ ]:
class LMModel2(Module):
    def __init__(self, vocab_sz, n_hidden):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)  
        self.h_h = nn.Linear(n_hidden, n_hidden)     
        self.h_o = nn.Linear(n_hidden,vocab_sz)
        
    def forward(self, x):
        h = 0
        for i in range(3):
            h = h + self.i_h(x[:,i])
            h = F.relu(self.h_h(h))
        return self.h_o(h)

Verifichiamo di ottenere gli stessi risultati utilizzando questo refactoring:

In [ ]:
learn = Learner(dls, LMModel2(len(vocab), 64), loss_func=F.cross_entropy, 
                metrics=accuracy)
learn.fit_one_cycle(4, 1e-3)

epoch,train_loss,valid_loss,accuracy,time
0,1.816274,1.964143,0.460185,00:02
1,1.423805,1.739964,0.473259,00:02
2,1.430327,1.685172,0.485382,00:02
3,1.388390,1.657033,0.470406,00:02


Possiamo anche rifattorizzare la nostra rappresentazione pittorica esattamente nello stesso modo, come mostrato in <<basic_rnn>> (stiamo rimuovendo anche i dettagli delle dimensioni di attivazione qui e utilizzando gli stessi colori delle frecce di <<lm_rep>>).

<img alt="Rete neurale ricorrente di base" width="400" caption="Rete neurale ricorrente di base" id="basic_rnn" src="images/att_00070.png">

Vedrai che c'è una serie di attivazioni che vengono aggiornate ogni volta attraverso il ciclo, memorizzate nella variabile `h`: questo è chiamato *stato nascosto*.

> Gergo: stato nascosto: le attivazioni che vengono aggiornate ad ogni passaggio di una rete neurale ricorrente.

Una rete neurale definita utilizzando un loop come questo è chiamata *rete neurale ricorrente* (RNN). È importante comprendere che una RNN non è una nuova architettura complicata, ma semplicemente un refactoring di una rete neurale multistrato utilizzando un ciclo "for".

> R: La mia vera opinione: se venissero chiamate "looping neural network" o LNN, sembrerebbero il 50% meno scoraggianti!

Ora che sappiamo cos'è una RNN, proviamo a migliorarla un po'.

## Miglioramento della RNN

Osservando il codice del nostro RNN, una cosa che sembra problematica è che stiamo inizializzando il nostro stato nascosto a zero per ogni nuova sequenza di input. Perché è un problema? Abbiamo reso le nostre sequenze di campioni brevi in ​​modo che potessero essere facilmente inserite nei lotti. Ma se ordiniamo correttamente i campioni, tali sequenze verranno lette in ordine dal modello, esponendo il modello a lunghi tratti della sequenza originale. 

Un'altra cosa a cui possiamo prestare attenzione è avere più segnale: perché prevedere solo la quarta parola quando potremmo utilizzare le previsioni intermedie per prevedere anche la seconda e la terza parola? 

Vediamo come implementare tali modifiche, iniziando con l'aggiunta di alcuni stati.

### Mantenimento dello stato di una RNN

Poiché inizializziamo lo stato nascosto del modello su zero per ogni nuovo campione, stiamo buttando via tutte le informazioni che abbiamo sulle frasi che abbiamo visto finora, il che significa che il nostro modello in realtà non sa dove siamo nella sequenza di conteggio complessiva. Questo è facilmente risolvibile; possiamo semplicemente spostare l'inizializzazione dello stato nascosto su `__init__`.

Ma questa soluzione creerà un problema sottile, ma importante. Rende effettivamente la nostra rete neurale profonda quanto l'intero numero di token nel nostro documento. Ad esempio, se nel nostro set di dati fossero presenti 10.000 token, creeremmo una rete neurale a 10.000 livelli.

Per capire perché questo è il caso, consideriamo la rappresentazione pittorica originale della nostra rete neurale ricorrente in <<lm_rep>>, prima di refactoring con un ciclo `for`. Puoi vedere che ogni livello corrisponde a un input token. Quando parliamo della rappresentazione di una rete neurale ricorrente prima del refactoring con il ciclo "for", la chiamiamo *rappresentazione srotolata*. Spesso è utile considerare la rappresentazione srotolata quando si cerca di comprendere un RNN.

Il problema con una rete neurale a 10.000 strati è che se e quando arrivi alla decimillesima parola del set di dati, dovrai comunque calcolare le derivate fino al primo strato. Sarà davvero molto lento e richiederà molta memoria. È improbabile che tu possa memorizzare anche un mini-batch sulla tua GPU.

La soluzione a questo problema è dire a PyTorch che non vogliamo propagare all'indietro i derivati ​​​​attraverso l'intera rete neurale implicita. Invece, manterremo solo gli ultimi tre strati di gradienti. Per rimuovere tutta la cronologia del gradiente in PyTorch, utilizziamo il metodo `detach`.

Ecco la nuova versione del nostro RNN. Ora è stateful, perché ricorda le sue attivazioni tra le diverse chiamate a `forward`, che rappresentano il suo utilizzo per diversi campioni nel batch:

In [ ]:
class LMModel3(Module):
    def __init__(self, vocab_sz, n_hidden):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)  
        self.h_h = nn.Linear(n_hidden, n_hidden)     
        self.h_o = nn.Linear(n_hidden,vocab_sz)
        self.h = 0
        
    def forward(self, x):
        for i in range(3):
            self.h = self.h + self.i_h(x[:,i])
            self.h = F.relu(self.h_h(self.h))
        out = self.h_o(self.h)
        self.h = self.h.detach()
        return out
    
    def reset(self): self.h = 0

Questo modello avrà le stesse attivazioni qualunque sia la lunghezza della sequenza scelta, poiché lo stato nascosto ricorderà l'ultima attivazione del batch precedente. L'unica cosa che sarà diversa saranno i gradienti calcolati in ogni passaggio: verranno calcolati solo sui token della lunghezza della sequenza nel passato, anziché sull'intero flusso. Questo approccio è chiamato *backpropagation through time* (BPTT).

> gergo: Propagazione all'indietro nel tempo (BPTT): trattare una rete neurale con effettivamente uno strato per passaggio temporale (solitamente rifattorizzato utilizzando un ciclo) come un grande modello e calcolare i gradienti su di esso nel solito modo. Per evitare di rimanere senza memoria e tempo, di solito utilizziamo _truncated_ BPTT, che "stacca" la cronologia dei passaggi di calcolo nello stato nascosto ogni pochi passaggi temporali.

Per utilizzare "LMModel3", dobbiamo assicurarci che i campioni verranno visualizzati in un determinato ordine. Come abbiamo visto in <<chapter_nlp>>, se la prima riga del primo batch è il nostro `dset[0]`, allora il secondo batch dovrebbe avere `dset[1]` come prima riga, in modo che il modello veda il testo scorrere.

`LMDataLoader` lo faceva per noi in <<chapter_nlp>>. Questa volta lo faremo da soli.

Per fare ciò, riorganizzeremo il nostro set di dati. Per prima cosa dividiamo i campioni in gruppi `m = len(dset) // bs` (questo equivale a dividere l'intero set di dati concatenati, ad esempio, in 64 pezzi di uguali dimensioni, poiché qui stiamo usando `bs=64`). "m" è la lunghezza di ciascuno di questi pezzi. Ad esempio, se utilizziamo l'intero set di dati (anche se tra poco lo divideremo in treno e valido), sarà:

In [ ]:
m = len(seqs)//bs
m,bs,len(seqs)

(328, 64, 21031)

Il primo lotto sarà composto dai campioni:

    (0, m, 2*m, ..., (bs-1)*m)

il secondo lotto di campioni: 

    (1, m+1, 2*m+1, ..., (bs-1)*m+1)

e così via. In questo modo, ad ogni epoca, il modello vedrà un pezzo di testo contiguo di dimensione "3*m" (poiché ogni testo è di dimensione 3) su ogni riga del batch.

La seguente funzione esegue la reindicizzazione:

In [ ]:
def group_chunks(ds, bs):
    m = len(ds) // bs
    new_ds = L()
    for i in range(m): new_ds += L(ds[i + m*j] for j in range(bs))
    return new_ds

Quindi passiamo semplicemente `drop_last=True` durante la creazione dei nostri `DataLoaders` per eliminare l'ultimo batch che non ha la forma `bs`. Passiamo anche `shuffle=False` per assicurarci che i testi vengano letti in ordine:

In [ ]:
cut = int(len(seqs) * 0.8)
dls = DataLoaders.from_dsets(
    group_chunks(seqs[:cut], bs), 
    group_chunks(seqs[cut:], bs), 
    bs=bs, drop_last=True, shuffle=False)

L'ultima cosa che aggiungiamo è una piccola modifica al ciclo di allenamento tramite un `Callback`. Parleremo più approfonditamente delle callback in <<chapter_accel_sgd>>; questo chiamerà il metodo "reset" del nostro modello all'inizio di ogni epoca e prima di ogni fase di validazione. Poiché abbiamo implementato questo metodo per azzerare lo stato nascosto del modello, questo assicurerà di iniziare con uno stato pulito prima di leggere quei blocchi continui di testo. Possiamo anche iniziare ad allenarci un po' più a lungo:

In [ ]:
learn = Learner(dls, LMModel3(len(vocab), 64), loss_func=F.cross_entropy,
                metrics=accuracy, cbs=ModelResetter)
learn.fit_one_cycle(10, 3e-3)

epoch,train_loss,valid_loss,accuracy,time
0,1.677074,1.827367,0.467548,00:02
1,1.282722,1.870913,0.388942,00:02
2,1.090705,1.651793,0.462500,00:02
3,1.005092,1.613794,0.516587,00:02
4,0.965975,1.560775,0.551202,00:02
5,0.916182,1.595857,0.560577,00:02
6,0.897657,1.539733,0.574279,00:02
7,0.836274,1.585141,0.583173,00:02
8,0.805877,1.629808,0.586779,00:02
9,0.795096,1.651267,0.588942,00:02


Questo è già meglio! Il passo successivo è utilizzare più target e confrontarli con le previsioni intermedie.

### Creare più segnale

Un altro problema con il nostro approccio attuale è che prevediamo solo una parola di output per ogni tre parole di input. Ciò significa che la quantità di segnale che restituiamo per aggiornare i pesi non è così grande come potrebbe essere. Sarebbe meglio se prevedessimo la parola successiva dopo ogni singola parola, anziché ogni tre parole, come mostrato in <<stateful_rep>>.

<img alt="Previsione RNN dopo ogni token" width="400" caption="Previsione RNN dopo ogni token" id="stateful_rep" src="images/att_00024.png">

Questo è abbastanza facile da aggiungere. Dobbiamo prima modificare i nostri dati in modo che la variabile dipendente abbia ciascuna delle tre parole successive dopo ciascuna delle nostre tre parole di input. Invece di "3", utilizziamo un attributo, "sl" (per la lunghezza della sequenza) e lo rendiamo un po' più grande:

In [ ]:
sl = 16
seqs = L((tensor(nums[i:i+sl]), tensor(nums[i+1:i+sl+1]))
         for i in range(0,len(nums)-sl-1,sl))
cut = int(len(seqs) * 0.8)
dls = DataLoaders.from_dsets(group_chunks(seqs[:cut], bs),
                             group_chunks(seqs[cut:], bs),
                             bs=bs, drop_last=True, shuffle=False)

Osservando il primo elemento di `seqs`, possiamo vedere che contiene due elenchi della stessa dimensione. Il secondo elenco è uguale al primo, ma sfalsato di un elemento:

In [ ]:
[L(vocab[o] for o in s) for s in seqs[0]]

[(#16) ['one','.','two','.','three','.','four','.','five','.'...],
 (#16) ['.','two','.','three','.','four','.','five','.','six'...]]

Ora dobbiamo modificare il nostro modello in modo che restituisca una previsione dopo ogni parola, anziché solo alla fine di una sequenza di tre parole:

In [ ]:
class LMModel4(Module):
    def __init__(self, vocab_sz, n_hidden):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)  
        self.h_h = nn.Linear(n_hidden, n_hidden)     
        self.h_o = nn.Linear(n_hidden,vocab_sz)
        self.h = 0
        
    def forward(self, x):
        outs = []
        for i in range(sl):
            self.h = self.h + self.i_h(x[:,i])
            self.h = F.relu(self.h_h(self.h))
            outs.append(self.h_o(self.h))
        self.h = self.h.detach()
        return torch.stack(outs, dim=1)
    
    def reset(self): self.h = 0

Questo modello restituirà output di forma `bs x sl x vocab_sz` (poiché abbiamo impilato su `dim=1`). I nostri obiettivi hanno la forma `bs x sl`, quindi dobbiamo appiattirli prima di usarli in `F.cross_entropy`:

In [ ]:
def loss_func(inp, targ):
    return F.cross_entropy(inp.view(-1, len(vocab)), targ.view(-1))

Ora possiamo utilizzare questa funzione di perdita per addestrare il modello:

In [ ]:
learn = Learner(dls, LMModel4(len(vocab), 64), loss_func=loss_func,
                metrics=accuracy, cbs=ModelResetter)
learn.fit_one_cycle(15, 3e-3)

epoch,train_loss,valid_loss,accuracy,time
0,3.103298,2.874341,0.212565,00:01
1,2.231964,1.971280,0.462158,00:01
2,1.711358,1.813547,0.461182,00:01
3,1.448516,1.828176,0.483236,00:01
4,1.288630,1.659564,0.520671,00:01
5,1.161470,1.714023,0.554932,00:01
6,1.055568,1.660916,0.575033,00:01
7,0.960765,1.719624,0.591064,00:01
8,0.870153,1.839560,0.614665,00:01
9,0.808545,1.770278,0.624349,00:01


Dobbiamo allenarci più a lungo, perché il compito è un po’ cambiato ed ora è più complicato. Ma alla fine il risultato è buono... Almeno, a volte. Se lo esegui alcune volte, vedrai che puoi ottenere risultati abbastanza diversi su esecuzioni diverse. Questo perché in effetti qui abbiamo una rete molto profonda, che può risultare in gradienti molto grandi o molto piccoli. Vedremo nella prossima parte di questo capitolo come affrontare questo problema.

Ora, il modo più ovvio per ottenere un modello migliore è andare più in profondità: abbiamo solo uno strato lineare tra lo stato nascosto e le attivazioni dell'output nella nostra RNN di base, quindi forse otterremo risultati migliori con di più.

## RNN multistrato

In una RNN multistrato, passiamo le attivazioni dalla nostra rete neurale ricorrente a una seconda rete neurale ricorrente, come in <<stacked_rnn_rep>>.

<img alt="RNN a 2 strati" width="550" caption="RNN a 2 strati" id="stacked_rnn_rep" src="images/att_00025.png">

La rappresentazione srotolata è mostrata in <<unrolled_stack_rep>> (simile a <<lm_rep>>).

<img alt="RNN srotolato a due strati" width="500" caption="RNN srotolato a due strati" id="unrolled_stack_rep" src="images/att_00026.png">

Vediamo come implementare questo nella pratica.

### Il modello

Possiamo risparmiare tempo utilizzando la classe "RNN" di PyTorch, che implementa esattamente ciò che abbiamo creato in precedenza, ma ci dà anche la possibilità di impilare più RNN, come abbiamo discusso:

In [ ]:
class LMModel5(Module):
    def __init__(self, vocab_sz, n_hidden, n_layers):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)
        self.rnn = nn.RNN(n_hidden, n_hidden, n_layers, batch_first=True)
        self.h_o = nn.Linear(n_hidden, vocab_sz)
        self.h = torch.zeros(n_layers, bs, n_hidden)
        
    def forward(self, x):
        res,h = self.rnn(self.i_h(x), self.h)
        self.h = h.detach()
        return self.h_o(res)
    
    def reset(self): self.h.zero_()

In [ ]:
learn = Learner(dls, LMModel5(len(vocab), 64, 2), 
                loss_func=CrossEntropyLossFlat(), 
                metrics=accuracy, cbs=ModelResetter)
learn.fit_one_cycle(15, 3e-3)

epoch,train_loss,valid_loss,accuracy,time
0,3.055853,2.591640,0.437907,00:01
1,2.162359,1.787310,0.471598,00:01
2,1.710663,1.941807,0.321777,00:01
3,1.520783,1.999726,0.312012,00:01
4,1.330846,2.012902,0.413249,00:01
5,1.163297,1.896192,0.450684,00:01
6,1.033813,2.005209,0.434814,00:01
7,0.919090,2.047083,0.456706,00:01
8,0.822939,2.068031,0.468831,00:01
9,0.750180,2.136064,0.475098,00:01


Questo è deludente... il nostro precedente RNN a strato singolo ha funzionato meglio. Perché? Il motivo è che abbiamo un modello più profondo, che porta ad attivazioni che esplodono o svaniscono.

### Attivazioni esplosive o scomparse

In pratica, creare modelli accurati da questo tipo di RNN è difficile. Otterremo risultati migliori se chiamiamo "stacca" meno spesso e disponiamo di più livelli: questo offre alla nostra RNN un orizzonte temporale più lungo da cui imparare e funzionalità più ricche da creare. Ma significa anche che abbiamo un modello più profondo da formare. La sfida chiave nello sviluppo del deep learning è stata capire come addestrare questo tipo di modelli.

Il motivo per cui questo è difficile è a causa di ciò che accade quando moltiplichi per una matrice molte volte. Pensa a cosa succede quando moltiplichi per un numero molte volte. Ad esempio, se moltiplichi per 2, partendo da 1, ottieni la sequenza 1, 2, 4, 8,... dopo 32 passi sei già a 4.294.967.296. Un problema simile accade se moltiplichi per 0,5: ottieni 0,5, 0,25, 0,125... e dopo 32 passaggi è 0,00000000023. Come puoi vedere, moltiplicare per un numero anche leggermente superiore o inferiore a 1 provoca un'esplosione o una scomparsa del nostro numero di partenza, dopo solo poche moltiplicazioni ripetute.

Poiché la moltiplicazione di matrici consiste semplicemente nel moltiplicare i numeri e sommarli, esattamente la stessa cosa accade con le moltiplicazioni di matrici ripetute. E una rete neurale profonda è proprio questo: ogni strato aggiuntivo è un'altra moltiplicazione di matrice. Ciò significa che è molto facile per una rete neurale profonda ritrovarsi con numeri estremamente grandi o estremamente piccoli.

Questo è un problema, perché il modo in cui i computer memorizzano i numeri (noto come "virgola mobile") significa che diventano sempre meno accurati quanto più i numeri si allontanano dallo zero. Il diagramma in <<float_prec>>, dall'eccellente articolo ["Quello che non avresti mai voluto sapere sulla virgola mobile ma sarai costretto a scoprirlo"](http://www.volkerschatz.com/science/float.html), mostra come la precisione dei numeri in virgola mobile varia sulla linea numerica.

<img alt="Precisione dei numeri in virgola mobile" width="1000" caption="Precisione dei numeri in virgola mobile" id="float_prec" src="images/fltscale.svg">

Questa imprecisione fa sì che spesso i gradienti calcolati per l'aggiornamento dei pesi risultino pari a zero o infinito per le reti profonde. Questo è comunemente chiamato problema dei *gradienti che svaniscono* o del *gradienti che esplodono*. Ciò significa che in SGD i pesi non vengono aggiornati affatto o saltano all'infinito. In ogni caso, non miglioreranno con l'allenamento.

I ricercatori hanno sviluppato diversi modi per affrontare questo problema, di cui parleremo più avanti nel libro. Un'opzione è modificare la definizione di un livello in modo da rendere meno probabile che si verifichino attivazioni esplosive. Esamineremo i dettagli di come ciò viene fatto in <<chapter_convolutions>>, quando discutiamo della normalizzazione batch, e in <<chapter_resnet>>, quando discutiamo di ResNets, sebbene questi dettagli generalmente non abbiano importanza nella pratica (a meno che tu non sia un ricercatore che sta creando nuovi approcci per risolvere questo problema). Un'altra strategia per affrontare questo problema è fare attenzione all'inizializzazione, che è un argomento che esamineremo in <<chapter_foundations>>.

Per le RNN, esistono due tipi di livelli che vengono frequentemente utilizzati per evitare attivazioni esplosive: *unità ricorrenti gate* (GRU) e strati di *memoria a breve termine* (LSTM). Entrambi sono disponibili in PyTorch e sono sostitutivi del livello RNN. In questo libro tratteremo solo gli LSTM; ci sono molti buoni tutorial online che spiegano i GRU, che sono una variante minore del design LSTM.

##LSTM

LSTM è un'architettura introdotta nel 1997 da Jürgen Schmidhuber e Sepp Hochreiter. In questa architettura non ci sono uno ma due stati nascosti. Nella nostra RNN di base, lo stato nascosto è l'output della RNN nella fase temporale precedente. Questo stato nascosto è quindi responsabile di due cose:

- Avere le informazioni giuste affinché il livello di output possa prevedere il token successivo corretto
- Conservare memoria di tutto ciò che è accaduto nella frase

Consideriamo, ad esempio, le frasi "Henry ha un cane e gli piace molto il suo cane" e "Sophie ha un cane e le piace molto il suo cane". È molto chiaro che l'RNN deve ricordare il nome all'inizio della frase per poter prevedere *lui/lei* o *suo/lei*. 

In pratica, gli RNN sono davvero pessimi nel conservare la memoria di ciò che è accaduto molto prima nella frase, che è la motivazione per avere un altro stato nascosto (chiamato *stato della cella*) nell'LSTM. Lo stato della cella sarà responsabile del mantenimento della *memoria a breve termine*, mentre lo stato nascosto si concentrerà sul prossimo token da prevedere. Diamo uno sguardo più da vicino a come si ottiene questo risultato e costruiamo un LSTM da zero.

### Costruire un LSTM da zero

Per costruire un LSTM, dobbiamo prima comprenderne l'architettura. <<lstm>> mostra la sua struttura interna.
    
<img src="images/LSTM.png" id="lstm" caption="Architettura di un LSTM" alt="Un grafico che mostra l'architettura interna di un LSTM" width="700">

In questa immagine, il nostro input $x_{t}$ entra a sinistra con lo stato nascosto precedente ($h_{t-1}$) e lo stato della cella ($c_{t-1}$). I quattro riquadri arancioni rappresentano quattro strati (le nostre reti neurali) con l'attivazione che può essere sigmoide ($\sigma$) o tanh. tanh è semplicemente una funzione sigmoide scalata nell'intervallo da -1 a 1. La sua espressione matematica può essere scritta in questo modo:

$$\tanh(x) = \frac{e^{x} - e^{-x}}{e^{x}+e^{-x}} = 2 \sigma(2x) - 1$$

dove $\sigma$ è la funzione sigmoide. I cerchi verdi sono operazioni elementari. Ciò che appare a destra è il nuovo stato nascosto ($h_{t}$) e il nuovo stato della cella ($c_{t}$), pronti per il nostro prossimo input. Il nuovo stato nascosto viene utilizzato anche come output, motivo per cui la freccia si divide per andare verso l'alto.

Esaminiamo le quattro reti neurali (chiamate *porte*) una per una e spieghiamo il diagramma, ma prima di ciò, notiamo quanto poco sia cambiato lo stato della cella (in alto). Non passa nemmeno direttamente attraverso una rete neurale! Questo è esattamente il motivo per cui porterà avanti uno stato a lungo termine.

Innanzitutto, le frecce per l'input e il vecchio stato nascosto vengono unite. Nella RNN che abbiamo scritto in precedenza in questo capitolo, li stavamo sommando. Nell'LSTM li impiliamo in un unico grande tensore. Ciò significa che la dimensione dei nostri incorporamenti (che è la dimensione di $x_{t}$) può essere diversa dalla dimensione del nostro stato nascosto. Se li chiamiamo `n_in` e `n_hid`, la freccia in basso avrà le dimensioni di `n_in + n_hid`; quindi tutte le reti neurali (riquadri arancioni) sono strati lineari con input `n_in + n_hid` e output `n_hid`.

Il primo cancello (guardando da sinistra a destra) è chiamato *cancello del dimenticatoio*. Poiché si tratta di uno strato lineare seguito da un sigmoide, il suo output sarà costituito da scalari compresi tra 0 e 1. Moltiplichiamo questo risultato per lo stato della cella per determinare quali informazioni conservare e quali eliminare: i valori più vicini a 0 vengono scartati e i valori più vicini a 1 vengono mantenuti. Ciò conferisce all'LSTM la capacità di dimenticare cose sul suo stato a lungo termine. Ad esempio, quando si attraversa un punto o un token "xxbos", ci si aspetterebbe che (avesse imparato a) reimpostare lo stato della cella.

La seconda porta è chiamata *porta di ingresso*. Funziona con la terza porta (che in realtà non ha un nome ma a volte viene chiamata *porta della cella*) per aggiornare lo stato della cella. Ad esempio, potremmo vedere un nuovo pronome di genere, nel qual caso dovremo sostituire le informazioni sul genere rimosse dal cancello del dimenticatoio. Similmente alla porta di dimenticanza, la porta di input decide quali elementi dello stato della cella aggiornare (valori vicini a 1) o meno (valori vicini a 0). La terza porta determina quali sono i valori aggiornati, nell'intervallo da –1 a 1 (grazie alla funzione tanh). Il risultato viene quindi aggiunto allo stato della cella.

L'ultima porta è la *porta di uscita*. Determina quali informazioni dallo stato della cella utilizzare per generare l'output. Lo stato della cella passa attraverso un tanh prima di essere combinato con l'output sigmoideo dal gate di output e il risultato è il nuovo stato nascosto.

In termini di codice, possiamo scrivere gli stessi passaggi in questo modo:

In [ ]:
class LSTMCell(Module):
    def __init__(self, ni, nh):
        self.forget_gate = nn.Linear(ni + nh, nh)
        self.input_gate  = nn.Linear(ni + nh, nh)
        self.cell_gate   = nn.Linear(ni + nh, nh)
        self.output_gate = nn.Linear(ni + nh, nh)

    def forward(self, input, state):
        h,c = state
        h = torch.cat([h, input], dim=1)
        forget = torch.sigmoid(self.forget_gate(h))
        c = c * forget
        inp = torch.sigmoid(self.input_gate(h))
        cell = torch.tanh(self.cell_gate(h))
        c = c + inp * cell
        out = torch.sigmoid(self.output_gate(h))
        h = out * torch.tanh(c)
        return h, (h,c)

In pratica, possiamo quindi rifattorizzare il codice. Inoltre, in termini di prestazioni, è meglio fare una moltiplicazione di matrici grandi piuttosto che quattro più piccole (questo perché lanciamo lo speciale kernel veloce sulla GPU solo una volta e dà alla GPU più lavoro da svolgere in parallelo). L'impilamento richiede un po' di tempo (poiché dobbiamo spostare uno dei tensori sulla GPU per avere tutto in un array contiguo), quindi utilizziamo due livelli separati per l'input e lo stato nascosto. Il codice ottimizzato e rifattorizzato si presenta così:

In [ ]:
class LSTMCell(Module):
    def __init__(self, ni, nh):
        self.ih = nn.Linear(ni,4*nh)
        self.hh = nn.Linear(nh,4*nh)

    def forward(self, input, state):
        h,c = state
        # One big multiplication for all the gates is better than 4 smaller ones
        gates = (self.ih(input) + self.hh(h)).chunk(4, 1)
        ingate,forgetgate,outgate = map(torch.sigmoid, gates[:3])
        cellgate = gates[3].tanh()

        c = (forgetgate*c) + (ingate*cellgate)
        h = outgate * c.tanh()
        return h, (h,c)

Qui utilizziamo il metodo PyTorch `chunk` per dividere il nostro tensore in quattro parti. Funziona così:

In [ ]:
t = torch.arange(0,10); t

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [ ]:
t.chunk(2)

(tensor([0, 1, 2, 3, 4]), tensor([5, 6, 7, 8, 9]))

Usiamo ora questa architettura per addestrare un modello linguistico!

### Addestramento di un modello linguistico utilizzando LSTM

Ecco la stessa rete di "LMModel5", che utilizza un LSTM a due livelli. Possiamo addestrarlo a un tasso di apprendimento più elevato, per un tempo più breve e ottenere una migliore precisione:

In [ ]:
class LMModel6(Module):
    def __init__(self, vocab_sz, n_hidden, n_layers):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)
        self.rnn = nn.LSTM(n_hidden, n_hidden, n_layers, batch_first=True)
        self.h_o = nn.Linear(n_hidden, vocab_sz)
        self.h = [torch.zeros(n_layers, bs, n_hidden) for _ in range(2)]
        
    def forward(self, x):
        res,h = self.rnn(self.i_h(x), self.h)
        self.h = [h_.detach() for h_ in h]
        return self.h_o(res)
    
    def reset(self): 
        for h in self.h: h.zero_()

In [ ]:
learn = Learner(dls, LMModel6(len(vocab), 64, 2), 
                loss_func=CrossEntropyLossFlat(), 
                metrics=accuracy, cbs=ModelResetter)
learn.fit_one_cycle(15, 1e-2)

epoch,train_loss,valid_loss,accuracy,time
0,3.000821,2.663942,0.438314,00:02
1,2.139642,2.184780,0.240479,00:02
2,1.607275,1.812682,0.439779,00:02
3,1.347711,1.830982,0.497477,00:02
4,1.123113,1.937766,0.594401,00:02
5,0.852042,2.012127,0.631592,00:02
6,0.565494,1.312742,0.725749,00:02
7,0.347445,1.297934,0.711263,00:02
8,0.208191,1.441269,0.731201,00:02
9,0.126335,1.569952,0.737305,00:02


Questo è meglio di un RNN multistrato! Tuttavia, possiamo ancora notare un leggero eccesso di adattamento, il che indica che un po’ di regolarizzazione potrebbe essere d’aiuto.

## Regolarizzazione di un LSTM

Le reti neurali ricorrenti, in generale, sono difficili da addestrare, a causa del problema delle attivazioni e dei gradienti evanescenti che abbiamo visto prima. L'uso delle celle LSTM (o GRU) rende l'addestramento più semplice rispetto alle RNN vanilla, ma sono comunque molto inclini all'overfitting. L'aumento dei dati, pur essendo una possibilità, viene utilizzato meno spesso per i dati di testo che per le immagini perché nella maggior parte dei casi richiede un altro modello per generare aumenti casuali (ad esempio, traducendo il testo in un'altra lingua e poi di nuovo nella lingua originale). Nel complesso, l’aumento dei dati per i dati di testo non è attualmente uno spazio ben esplorato.

Tuttavia, esistono altre tecniche di regolarizzazione che possiamo utilizzare per ridurre l'overfitting, che sono state studiate attentamente per l'uso con gli LSTM nel documento ["Regularizing and Optimizing LSTM Language Models"](https://arxiv.org/abs/1708.02182) di Stephen Merity, Nitish Shirish Keskar e Richard Socher. Questo articolo ha mostrato come l'uso efficace di *dropout*, *regolarizzazione dell'attivazione* e *regolarizzazione dell'attivazione temporale* potrebbe consentire a un LSTM di superare risultati all'avanguardia che in precedenza richiedevano modelli molto più complicati. Gli autori hanno chiamato un LSTM che utilizza queste tecniche *AWD-LSTM*. Esamineremo ciascuna di queste tecniche a turno.

### Ritirarsi

Il dropout è una tecnica di regolarizzazione introdotta da Geoffrey Hinton et al. in [Migliorare le reti neurali impedendo il co-adattamento dei rilevatori di caratteristiche](https://arxiv.org/abs/1207.0580). L'idea di base è quella di modificare in modo casuale alcune attivazioni a zero al momento dell'allenamento. Ciò garantisce che tutti i neuroni lavorino attivamente verso l'output, come visto in <<img_dropout>> (da "Dropout: A Simple Way to Prevent Neural Networks from Overfitting" di Nitish Srivastava et al.).

<img src="images/Dropout1.png" alt="Una figura dall'articolo che mostra come i neuroni si attivano con il dropout" width="800" id="img_dropout" caption="Applicazione del dropout in una rete neurale (per gentile concessione di Nitish Srivastava et al.)">

Hinton ha usato una bella metafora quando ha spiegato, in un'intervista, l'ispirazione per l'abbandono scolastico:

> : Sono andato alla mia banca. I cassieri continuavano a cambiare e ho chiesto a uno di loro perché. Ha detto che non lo sapeva, ma si sono spostati molto. Ho pensato che fosse perché sarebbe stata necessaria la cooperazione tra i dipendenti per frodare con successo la banca. Ciò mi ha fatto capire che la rimozione casuale di un diverso sottoinsieme di neuroni in ciascun esempio avrebbe impedito le cospirazioni e quindi ridotto l'overfitting.

Nella stessa intervista, ha anche spiegato che le neuroscienze hanno fornito ulteriore ispirazione:

> : Non sappiamo veramente perché i neuroni aumentano. Una teoria è che vogliono essere rumorosi per regolarizzare, perché abbiamo molti più parametri che punti dati. L'idea del dropout è che se hai attivazioni rumorose, puoi permetterti di utilizzare un modello molto più grande.

Questo spiega l’idea alla base del perché il dropout aiuta a generalizzare: prima aiuta i neuroni a cooperare meglio insieme, poi rende le attivazioni più rumorose, rendendo così il modello più robusto.

Possiamo vedere, tuttavia, che se azzerassimo semplicemente quelle attivazioni senza fare nient'altro, il nostro modello avrebbe problemi di addestramento: se passiamo dalla somma di cinque attivazioni (che sono tutti numeri positivi poiché applichiamo una ReLU) a sole due, questa non avrà la stessa scala. Pertanto, se applichiamo il dropout con probabilità `p`, riscaliamo tutte le attivazioni dividendole per `1-p` (in media `p` verrà azzerato, quindi lascia `1-p`), come mostrato in <<img_dropout1>>.

<img src="images/Dropout.png" alt="Una figura dall'articolo che introduce il dropout che mostra come un neurone è acceso/spento" width="600" id="img_dropout1" caption="Perché ridimensionare le attivazioni quando si applica il dropout (per gentile concessione di Nitish Srivastava et al.)">

Questa è un'implementazione completa del livello di esclusione in PyTorch (sebbene il livello nativo di PyTorch sia in realtà scritto in C, non in Python):

In [ ]:
class Dropout(Module):
    def __init__(self, p): self.p = p
    def forward(self, x):
        if not self.training: return x
        mask = x.new(*x.shape).bernoulli_(1-p)
        return x * mask.div_(1-p)

Il metodo `bernoulli_` crea un tensore di zeri casuali (con probabilità "p") e uno (con probabilità "1-p"), che viene poi moltiplicato con il nostro input prima di dividere per "1-p". Da notare l'uso dell'attributo `training`, che è disponibile in qualsiasi PyTorch `nn.Module`, e ci dice se stiamo facendo training o inferenza.

> nota: fai i tuoi esperimenti: nei capitoli precedenti del libro aggiungeremo qui un esempio di codice per `bernoulli_`, così puoi vedere esattamente come funziona. Ma ora che ne sai abbastanza per farlo da solo, faremo sempre meno esempi per te e ci aspetteremo invece che tu faccia i tuoi esperimenti per vedere come funzionano le cose. In questo caso, vedrai nel questionario di fine capitolo che ti stiamo chiedendo di sperimentare `bernoulli_`, ma non aspettare che ti chiediamo di sperimentare per sviluppare la tua comprensione del codice che stiamo studiando; vai avanti e fallo comunque!

L'uso del dropout prima di passare l'output del nostro LSTM allo strato finale aiuterà a ridurre l'overfitting. Dropout è utilizzato anche in molti altri modelli, inclusa la testata CNN predefinita utilizzata in `fastai.vision`, ed è disponibile in `fastai.tabular` passando il parametro `ps` (dove ogni "p" viene passata a ciascun livello `Dropout` aggiunto), come vedremo in <<chapter_arch_details>>.

Dropout ha un comportamento diverso in modalità training e validazione, che abbiamo specificato utilizzando l'attributo "training" in "Dropout". Chiamare il metodo "train" su un "Modulo" imposta "training" su "True" (sia per il modulo su cui chiami il metodo sia per ogni modulo che contiene ricorsivamente) e "eval" lo imposta su "False". Questo viene fatto automaticamente quando si chiamano i metodi di `Learner`, ma se non si utilizza quella classe, ricordarsi di passare dall'uno all'altro secondo necessità.

### Regolarizzazione dell'attivazione e Regolarizzazione dell'attivazione temporale

La *regolarizzazione dell'attivazione* (AR) e la *regolarizzazione dell'attivazione temporale* (TAR) sono due metodi di regolarizzazione molto simili al decadimento del peso, discussi in <<chapter_collab>>. Quando applichiamo il decadimento del peso, aggiungiamo una piccola penalità alla perdita che mira a rendere i pesi il più piccoli possibile. Per la regolarizzazione dell'attivazione, cercheremo di ridurre il più possibile le attivazioni finali prodotte dall'LSTM, anziché i pesi.

Per regolarizzare le attivazioni finali, dobbiamo memorizzarle da qualche parte, quindi aggiungere la media dei loro quadrati alla perdita (insieme a un moltiplicatore "alpha", che è proprio come "wd" per il decadimento del peso):

``` pitone
perdita += alfa * attivazioni.pow(2).mean()
```

La regolarizzazione dell'attivazione temporale è legata al fatto che stiamo prevedendo dei token in una frase. Ciò significa che è probabile che i risultati dei nostri LSTM abbiano in qualche modo senso quando li leggiamo in ordine. TAR è lì per incoraggiare questo comportamento aggiungendo una penalità alla perdita per rendere la differenza tra due attivazioni consecutive la più piccola possibile: il nostro tensore di attivazioni ha una forma `bs x sl x n_hid`, e leggiamo le attivazioni consecutive sull'asse della lunghezza della sequenza (la dimensione al centro). Con questo, il TAR può essere espresso come:

``` pitone
perdita += beta * (attivazioni[:,1:] - attivazioni[:,:-1]).pow(2).mean()
```

"alpha" e "beta" sono quindi due iperparametri da ottimizzare. Per far sì che funzioni, abbiamo bisogno che il nostro modello con dropout restituisca tre cose: l'output corretto, le attivazioni dell'LSTM pre-dropout e le attivazioni dell'LSTM post-dropout. L'AR viene spesso applicato alle attivazioni abbandonate (per non penalizzare le attivazioni che successivamente abbiamo trasformato in zeri) mentre il TAR viene applicato alle attivazioni non abbandonate (perché quegli zeri creano grandi differenze tra due passaggi temporali consecutivi). C'è quindi un callback chiamato "RNNRegularizer" che applicherà questa regolarizzazione per noi.

### Allenamento di un LSTM regolarizzato con peso legato

Possiamo combinare il dropout (applicato prima di entrare nel nostro livello di output) con AR e TAR per addestrare il nostro precedente LSTM. Dobbiamo solo restituire tre cose invece di una: l'output normale del nostro LSTM, le attivazioni abbandonate e le attivazioni dei nostri LSTM. Gli ultimi due verranno prelevati dal callback "RNNRegularization" per i contributi che deve apportare alla perdita.

Un altro trucco utile che possiamo aggiungere dal [documento AWD LSTM](https://arxiv.org/abs/1708.02182) è *il peso vincolato*. In un modello linguistico, gli incorporamenti di input rappresentano una mappatura dalle parole inglesi alle attivazioni e lo strato nascosto di output rappresenta una mappatura dalle attivazioni alle parole inglesi. Potremmo aspettarci, intuitivamente, che queste mappature possano essere le stesse. Possiamo rappresentarlo in PyTorch assegnando la stessa matrice di peso a ciascuno di questi livelli:

    self.h_o.peso = self.i_h.peso

In "LMModel7" includiamo queste modifiche finali:

In [ ]:
class LMModel7(Module):
    def __init__(self, vocab_sz, n_hidden, n_layers, p):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)
        self.rnn = nn.LSTM(n_hidden, n_hidden, n_layers, batch_first=True)
        self.drop = nn.Dropout(p)
        self.h_o = nn.Linear(n_hidden, vocab_sz)
        self.h_o.weight = self.i_h.weight
        self.h = [torch.zeros(n_layers, bs, n_hidden) for _ in range(2)]
        
    def forward(self, x):
        raw,h = self.rnn(self.i_h(x), self.h)
        out = self.drop(raw)
        self.h = [h_.detach() for h_ in h]
        return self.h_o(out),raw,out
    
    def reset(self): 
        for h in self.h: h.zero_()

Possiamo creare uno `Learner` regolarizzato utilizzando il callback `RNNRegularizer`:

In [ ]:
learn = Learner(dls, LMModel7(len(vocab), 64, 2, 0.5),
                loss_func=CrossEntropyLossFlat(), metrics=accuracy,
                cbs=[ModelResetter, RNNRegularizer(alpha=2, beta=1)])

Un `TextLearner` aggiunge automaticamente questi due callback per noi (con quei valori per `alpha` e `beta` come predefiniti), quindi possiamo semplificare la riga precedente in:

In [ ]:
learn = TextLearner(dls, LMModel7(len(vocab), 64, 2, 0.4),
                    loss_func=CrossEntropyLossFlat(), metrics=accuracy)

Possiamo quindi addestrare il modello e aggiungere ulteriore regolarizzazione aumentando il decadimento del peso a "0.1":

In [ ]:
learn.fit_one_cycle(15, 1e-2, wd=0.1)

epoch,train_loss,valid_loss,accuracy,time
0,2.693885,2.013484,0.466634,00:02
1,1.685549,1.187310,0.629313,00:02
2,0.973307,0.791398,0.745605,00:02
3,0.555823,0.640412,0.794108,00:02
4,0.351802,0.557247,0.836100,00:02
5,0.244986,0.594977,0.807292,00:02
6,0.192231,0.511690,0.846761,00:02
7,0.162456,0.520370,0.858073,00:02
8,0.142664,0.525918,0.842285,00:02
9,0.128493,0.495029,0.858073,00:02


Ora questo è molto meglio del nostro modello precedente!

## Conclusione

Ora hai visto tutto ciò che c'è all'interno dell'architettura AWD-LSTM che abbiamo utilizzato nella classificazione del testo in <<chapter_nlp>>. Utilizza il dropout in molti più posti:

- Embedding dropout (all'interno del livello di incorporamento, rilascia alcune linee casuali di incorporamenti)
- Eliminazione dell'input (applicato dopo il livello di incorporamento)
- Perdita di peso (applicata ai pesi dell'LSTM ad ogni fase di allenamento)
- Dropout nascosto (applicato allo stato nascosto tra due livelli)

Ciò lo rende ancora più regolarizzato. Poiché la regolazione fine di questi cinque valori di esclusione (incluso l'esclusione prima del livello di output) è complicata, abbiamo determinato buoni valori predefiniti e consentiamo la regolazione complessiva dell'entità dell'interruzione con il parametro "drop_mult" che hai visto in quel capitolo (che viene moltiplicato per ogni interruzione).

Un'altra architettura molto potente, soprattutto nei problemi "sequenza-sequenza" (vale a dire, problemi in cui la variabile dipendente è essa stessa una sequenza di lunghezza variabile, come la traduzione linguistica), è l'architettura Transformers. Puoi trovarlo in un capitolo bonus sul [sito web del libro](https://book.fast.ai/).

## Questionario

1. Se il set di dati per il tuo progetto è così grande e complicato che lavorarci richiede una notevole quantità di tempo, cosa dovresti fare?
1. Perché concateniamo i documenti nel nostro set di dati prima di creare un modello linguistico?
1. Per utilizzare una rete standard completamente connessa per prevedere la quarta parola date le tre parole precedenti, quali due modifiche dobbiamo apportare al nostro modello?
1. Come possiamo condividere una matrice di peso su più livelli in PyTorch?
1. Scrivi un modulo che preveda la terza parola date le due parole precedenti di una frase, senza sbirciare.
1. Cos'è una rete neurale ricorrente?
1. Cos'è lo "stato nascosto"?
1. Qual è l'equivalente dello stato nascosto in "LMModel1"?
1. Per mantenere lo stato in una RNN, perché è importante passare il testo al modello in ordine?
1. Cos'è una rappresentazione "srotolata" di una RNN?
1. Perché il mantenimento dello stato nascosto in una RNN può portare a problemi di memoria e prestazioni? Come risolviamo questo problema?
1. Cos'è il "BPTT"?
1. Scrivere il codice per stampare i primi batch del set di validazione, inclusa la conversione degli ID dei token in stringhe inglesi, come abbiamo mostrato per i batch di dati IMDb in <<chapter_nlp>>.
1. Cosa fa il callback `ModelResetter`? Perché ne abbiamo bisogno?
1. Quali sono gli svantaggi di prevedere solo una parola di output per ogni tre parole di input?
1. Perché abbiamo bisogno di una funzione di perdita personalizzata per `LMModel4`?
1. Perché l'addestramento di `LMModel4` è instabile?
1. Nella rappresentazione spiegata possiamo vedere che una rete neurale ricorrente ha in realtà molti strati. Allora perché dobbiamo impilare gli RNN per ottenere risultati migliori?
1. Disegna una rappresentazione di un RNN impilato (multistrato).
1. Perché dovremmo ottenere risultati migliori in una RNN se chiamiamo `detach` meno spesso? Perché ciò non potrebbe accadere nella pratica con un semplice RNN?
1. Perché una rete profonda può comportare attivazioni molto grandi o molto piccole? Perché è importante?
1. Nella rappresentazione dei numeri in virgola mobile di un computer, quali numeri sono i più precisi?
1. Perché i gradienti evanescenti impediscono l'allenamento?
1. Perché è utile avere due stati nascosti nell'architettura LSTM? Qual è lo scopo di ciascuno?
1. Come vengono chiamati questi due stati in un LSTM?
1. Cos'è il tanh e come è correlato al sigma?
1. Qual è lo scopo di questo codice in "LSTMCell": "h = torch.cat([h, input], dim=1)"
1. Cosa fa `chunk` in PyTorch?
1. Studia attentamente la versione rifattorizzata di "LSTMCell" per assicurarti di comprendere come e perché fa la stessa cosa della versione non rifattorizzata.
1. Perché possiamo utilizzare un tasso di apprendimento più elevato per `LMModel6`?
1. Quali sono le tre tecniche di regolarizzazione utilizzate in un modello AWD-LSTM?
1. Cos'è il "abbandono"?
1. Perché aumentiamo le attivazioni con il drop-out? Viene applicato durante l'addestramento, l'inferenza o entrambi?
1. Qual è lo scopo di questa riga di "Dropout": "if not self.training: return x"
1. Sperimenta "bernoulli_" per capire come funziona.
1. Come imposti il ​​tuo modello in modalità training in PyTorch? In modalità valutazione?
1. Scrivi l'equazione per la regolarizzazione dell'attivazione (in matematica o in codice, come preferisci). In cosa differisce dal decadimento ponderale?
1. Scrivi l'equazione per la regolarizzazione dell'attivazione temporale (in matematica o in codice, come preferisci). Perché non dovremmo usarlo per i problemi di visione artificiale?
1. Cos'è il "peso vincolato" in un modello linguistico?

### Ulteriori ricerche

1. In "LMModel2", perché "forward" può iniziare con "h=0"? Perché non è necessario dire `h=torch.zeros(...)`?
1. Scrivi il codice per un LSTM da zero (puoi fare riferimento a <<lstm>>).
1. Cerca su Internet l'architettura GRU e implementala da zero, quindi prova ad addestrare un modello. Vedi se riesci a ottenere risultati simili a quelli che abbiamo visto in questo capitolo. Confronta i tuoi risultati con quelli del modulo "GRU" integrato di PyTorch.
1. Dai un'occhiata al codice sorgente di AWD-LSTM in fastai e prova a mappare ciascuna riga di codice ai concetti mostrati in questo capitolo.